In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
tags = pd.read_csv(r'ml-latest\tags.csv')
ratings = pd.read_csv(r'ml-latest\ratings.csv')
movies = pd.read_csv(r'ml-latest\movies.csv')
links = pd.read_csv(r'ml-latest\links.csv')

In [28]:
ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,1225734739
1,1,110,4.0,1225865086
2,1,158,4.0,1225733503
3,1,260,4.5,1225735204
4,1,356,5.0,1225735119
...,...,...,...,...
33832157,330975,8340,2.0,1091583256
33832158,330975,8493,2.5,1091585709
33832159,330975,8622,4.0,1091581777
33832160,330975,8665,3.0,1091581765


Ersätter | med mellanslag mellan varje genre för att TfidfVectorizer ska hantera varje genre individuellt istället för grupperade med varandra:

In [22]:
movies['genres'] = movies['genres'].str.replace('|', ' ', regex=False)

Aggregerar på movieId och har med varje tag med mellanslag mellan varje så att TfidfVectorizer ska kunna hantera det korrekt:

In [23]:
tags = tags.groupby('movieId')['tag'].agg(lambda x: ' '.join(x.dropna().astype(str))).reset_index()

Lägger till tags i movies och använder how='left' för att inte bli av med några filmer eftersom att alla filmer inte har tags:

In [24]:
movies = movies.merge(tags, 'left')

Ersätter 33085 NaNs i tag-kolumnen med tomma strängar för att de inte ska påverka vid cosine similarity:

In [25]:
movies = movies.fillna('')

Lägger till en kolumn där genres och tags är samlade för att senare kunna bestämma om jag vill utgå från den samlade vektorn eller räkna deras vikt separat:

In [26]:
movies['genres+tags'] = movies['genres'] + ' ' + movies['tag']

Lägger till rating- och year-kolumnerna:

In [27]:
mean_ratings = ratings.groupby('movieId')['rating'].mean().reset_index()
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)')
movies = movies.merge(mean_ratings, 'left', 'movieId')

Droppar NaNs som fanns i de nyligen tillagda kolumnerna då de exempel jag undersökte av dem var väldigt obskyra titlar som endast hade lett till brus:

In [28]:
movies = movies.dropna().reset_index(drop=True)

Städar upp i titel-kolumnen för att många av titlarna är skrivna med början av titeln i slutet:

In [ ]:
movies['title'] = movies['title'].str.replace(r'^(.*),\s(The|A|An)\s(\(\d{4}\))$', r'\2 \1 \3', regex=True)

Använder TfidfVectorizer för att skapa vektorer för kolumnerna genres, tag och genres+tags för att senare användas med cosine_similarity:

In [29]:
vectorizer = TfidfVectorizer()
genres_vector = vectorizer.fit_transform(movies['genres'])
tags_vector = vectorizer.fit_transform(movies['tag'])
combined_vector = vectorizer.fit_transform(movies['genres+tags'])

In [22]:
movie = input('Ange titel eller sökord för vilken film vill du har rekommendationer baserat på: ')

words = movie.lower().split()
mask = pd.Series([all(word in title.lower() for word in words) for title in movies['title']])
alternatives = movies[mask]
while alternatives.empty:
    movie = input('Inga filmer hittades, försök igen: ')
    words = movie.lower().split()
    mask = pd.Series([all(word in title.lower() for word in words) for title in movies['title']])
    alternatives = movies[mask]
for i, title in enumerate(alternatives['title']):
    print(f'{i}: {title}')

0: Whatever (1998)
1: Whatever It Takes (2000)
2: Whatever Happened to Aunt Alice? (1969)
3: Whatever Happened to Harold Smith? (1999)
4: Whatever Works (2009)
5: Whatever Lola Wants (2007)
6: Barry Crimmins: Whatever Threatens You (2016)
7: Whatever (1999)
8: Carl Barron: Whatever Comes Next (2005)
9: Love or Whatever (2012)
10: Connor McDavid: Whatever it Takes (2020)


In [23]:
choice = input('Här är filmerna som matchar titeln du gav, ange siffran för den filmen du tänkte på: ')
choice_index = alternatives.iloc[int(choice)].name
choice_year = alternatives.iloc[int(choice)]['year']
choice_title = alternatives.iloc[int(choice)]['title']
choice_vector = combined_vector[int(choice_index)]

X = combined_vector
Y = choice_vector

similarity = cosine_similarity(X, Y).flatten()
sorted_indices = np.argsort(similarity)

KeyError: 'year'

In [21]:
rec_movies = sorted_indices[-6:-1]

movies['title'].loc[rec_movies]

NameError: name 'sorted_indices' is not defined

In [20]:

top50 = sorted_indices[-51:-1]

year_filtered = movies.loc[top50]
year_filtered = year_filtered[abs(year_filtered['year'].astype(int) - int(choice_year)) <= 10]
year_filtered





NameError: name 'sorted_indices' is not defined